# Configs

## Libraries

In [ ]:
!pip install -U ultralytics

In [ ]:
import os
import datetime
from pathlib import Path
from PIL import Image, ExifTags
import pandas as pd
from ultralytics import YOLO

In [ ]:
import logging

class SuppressDeprecationFilter(logging.Filter):
    def filter(self, record):
        return "'half' is deprecated" not in record.getMessage()

logging.getLogger("ultralytics").addFilter(SuppressDeprecationFilter())

## Mount Google collab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Run this in a notebook cell to copy files to local scratch space
!cp -r "/content/drive/MyDrive/datasets/Computer_Vision_Project/frames" "/content/local_video_stills"


In [ ]:
# Run this in a notebook cell to copy files to local scratch space
!cp -r "/content/drive/MyDrive/datasets/Computer_Vision_Project/test_1" "/content/local_video_stills2"


In [ ]:
# Run this in a notebook cell to copy files to local scratch space
!cp -r "/content/drive/MyDrive/datasets/Computer_Vision_Project/test_2" "/content/local_video_stills3"


# Source data

In [ ]:
# ==========================================
# 1. Configuration Layer
# ==========================================
SOURCE_FOLDER = "/content/local_video_stills"
SOURCE_FOLDER2 = "/content/local_video_stills2"
SOURCE_FOLDER3 = "/content/local_video_stills3"
MODEL_WEIGHTS = "yolo26m.pt"  # Change to your custom weights path later
# The subset of categories you care about (must match model's class names)
TARGET_CATEGORIES = ["person", "bicycle", "motorcycle", "car", "traffic light"]

In [ ]:
# ==========================================
# 2. Metadata Extraction
# ==========================================
def get_image_timestamp(file_path):
    """
    Attempts to get the timestamp from EXIF metadata.
    Falls back to OS modification time if EXIF is unavailable.
    """
    try:
        img = Image.open(file_path)
        exif_data = img._getexif()
        if exif_data:
            # 36867 is the EXIF tag for DateTimeOriginal
            for tag, value in exif_data.items():
                if tag == 36867:
                    return value
    except Exception as e:
        pass # Handle or log image opening errors if necessary

    # Fallback: OS modification time
    mtime = os.path.getmtime(file_path)
    return datetime.datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')

# Pipeline Def

In [ ]:
# ==========================================
# 3. Vision Pipeline Function
# ==========================================
def run_vision_pipeline(source_folder, weights, target_cats):
    print(f"Loading model: {weights}...")
    model = YOLO(weights)

    model_classes = model.names
    pipeline_results = []
    valid_extensions = {".jpg", ".jpeg", ".png"}
    folder_path = Path(source_folder)

    processed_count = 0
    total_inference_time = 0  # Track cumulative engine latency

    # --- FIX 1: Sort files alphabetically.
    # For chronological order, use: sorted(folder_path.iterdir(), key=os.path.getmtime)
    for img_path in sorted(folder_path.iterdir()):
        if img_path.suffix.lower() not in valid_extensions:
            continue

        if processed_count >= 1642:
            print(f"Reached test limit of {processed_count} images. Stopping.")
            break

        filename = img_path.name
        timestamp = get_image_timestamp(img_path)
        cat_counts = {cat: 0 for cat in target_cats}

        # --- FIX 2 & 4: Direct inference to T4 GPU and track metrics
        #results = model(img_path, device="cuda", verbose=False)[0]
        results = model(img_path, device="cuda", half=True, conf=0.15, verbose=False)[0]
        #Optimized performance

        # Extract direct hardware inference latency (in milliseconds)
        total_inference_time += results.speed['inference']

        # 4. Aggregation Logic
        for box in results.boxes:
            cls_id = int(box.cls[0])
            cls_name = model_classes[cls_id]

            if cls_name in cat_counts:
                cat_counts[cls_name] += 1

        # 5. Data Structuring
        row = [filename] + [cat_counts[cat] for cat in target_cats] + [timestamp]
        pipeline_results.append(row)

        processed_count += 1

    # --- FIX 4: Output performance metrics
    if processed_count > 0:
        avg_time = total_inference_time / processed_count
        print("\n==========================================")
        print(f"PERFORMANCE STATS:")
        print(f"Total Images Processed: {processed_count}")
        print(f"Average Inference Time: {avg_time:.2f} ms per image")
        print("==========================================\n")

    column_headers = ["filename"] + target_cats + ["timestamp"]
    df = pd.DataFrame(pipeline_results, columns=column_headers)

    return df

In [ ]:
# ==========================================
# 3.1 Logs Function (not in use)
# ==========================================

def log_experiment(log_drive_path, model_name, half_precision, conf_thresh, avg_time, total_images, df_results, target_cats):
    """
    Appends the parameters and summary metrics of the current run to a central CSV in Google Drive.
    """
    # 1. Gather all experimental metadata
    run_summary = {
        "timestamp": datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "model_weights": model_name,
        "half_precision": half_precision,
        "confidence_threshold": conf_thresh,
        "images_processed": total_images,
        "avg_inference_ms": round(avg_time, 2)
    }

    # 2. Aggregate total detections for each target category in this batch
    for cat in target_cats:
        run_summary[f"total_{cat}_detected"] = int(df_results[cat].sum())

    new_run_df = pd.DataFrame([run_summary])

    # 3. Append to the central CSV file in Google Drive
    if os.path.exists(log_drive_path):
        existing_df = pd.read_csv(log_drive_path)
        updated_df = pd.concat([existing_df, new_run_df], ignore_index=True)
        updated_df.to_csv(log_drive_path, index=False)
    else:
        new_run_df.to_csv(log_drive_path, index=False)

    print(f"\nExperiment metrics logged to Google Drive: {log_drive_path}")
    print(new_run_df.to_string(index=False))

# Main

In [ ]:
# ==========================================
# Run "Frames" Folder
# ==========================================
if __name__ == "__main__":
    # Ensure folder exists for the test run
    Path(SOURCE_FOLDER).mkdir(parents=True, exist_ok=True)

    results_df = run_vision_pipeline(SOURCE_FOLDER, MODEL_WEIGHTS, TARGET_CATEGORIES)

    print("\n--- Pipeline Execution Complete ---")
    print(results_df.head())

    OUTPUT_FOLDER = "/content/drive/MyDrive/datasets/"  # <-- set your folder here

    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    output_path = Path(OUTPUT_FOLDER) / "results1.csv"
    results_df.to_csv(output_path, index=False)
    print(f"Saved results to {output_path}")

    # From here, you can easily convert the numerical data to a PyTorch/NumPy Tensor:
    # tensor_data = results_df[TARGET_CATEGORIES].to_numpy()

Loading model: yolo26m.pt...
Reached test limit of 1642 images. Stopping.

PERFORMANCE STATS:
Total Images Processed: 1642
Average Inference Time: 14.37 ms per image


--- Pipeline Execution Complete ---
                   filename  person  bicycle  motorcycle  car  traffic light  \
0  DJI_0348_frame_00001.jpg       0        0           0   14              0   
1  DJI_0348_frame_00002.jpg       0        0           0    3              0   
2  DJI_0348_frame_00003.jpg       0        0           0   13              0   
3  DJI_0348_frame_00004.jpg       1        1           0    3              0   
4  DJI_0348_frame_00006.jpg       0        0           0   10              0   

             timestamp  
0  2026-07-17 13:58:59  
1  2026-07-17 13:58:59  
2  2026-07-17 13:58:59  
3  2026-07-17 13:58:59  
4  2026-07-17 13:58:59  
Saved results to /content/drive/MyDrive/datasets/results1.csv


In [ ]:
# ==========================================
# Run "Test_1" Folder
# ==========================================
if __name__ == "__main__":
    # Ensure folder exists for the test run
    Path(SOURCE_FOLDER2).mkdir(parents=True, exist_ok=True)

    results_df = run_vision_pipeline(SOURCE_FOLDER2, MODEL_WEIGHTS, TARGET_CATEGORIES)

    print("\n--- Pipeline Execution Complete ---")
    print(results_df.head())

    OUTPUT_FOLDER = "/content/drive/MyDrive/datasets/"  # <-- set your folder here

    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    output_path = Path(OUTPUT_FOLDER) / "results2.csv"
    results_df.to_csv(output_path, index=False)
    print(f"Saved results to {output_path}")

    # From here, you can easily convert the numerical data to a PyTorch/NumPy Tensor:
    # tensor_data = results_df[TARGET_CATEGORIES].to_numpy()

Loading model: yolo26m.pt...

PERFORMANCE STATS:
Total Images Processed: 1001
Average Inference Time: 14.45 ms per image


--- Pipeline Execution Complete ---
                   filename  person  bicycle  motorcycle  car  traffic light  \
0  DJI_0392_frame_00000.jpg       0        0           0    0              0   
1  DJI_0392_frame_00001.jpg       1        0           0    8              1   
2  DJI_0392_frame_00002.jpg       1        0           0    2              0   
3  DJI_0392_frame_00003.jpg       1        0           0    4              1   
4  DJI_0392_frame_00004.jpg       7        0           0    9              0   

             timestamp  
0  2026-07-17 14:07:34  
1  2026-07-17 14:06:07  
2  2026-07-17 14:06:07  
3  2026-07-17 14:06:08  
4  2026-07-17 14:06:08  
Saved results to /content/drive/MyDrive/datasets/results2.csv


In [ ]:
# ==========================================
# Run "Test_2" Folder
# ==========================================
#from datetime import datetime

if __name__ == "__main__":

    #CENTRAL_LOG_PATH = "/content/drive/MyDrive/datasets/cv_experiment_log.csv"
    #USE_HALF = True
    #CONF_THRESHOLD = 0.15
    # Ensure folder exists for the test run
    Path(SOURCE_FOLDER3).mkdir(parents=True, exist_ok=True)

    results_df = run_vision_pipeline(SOURCE_FOLDER3, MODEL_WEIGHTS, TARGET_CATEGORIES)

    print("\n--- Pipeline Execution Complete ---")
    print(results_df.head())



    OUTPUT_FOLDER = "/content/drive/MyDrive/datasets/"  # <-- set your folder here

    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    output_path = Path(OUTPUT_FOLDER) / "results.csv"
    results_df.to_csv(output_path, index=False)
    print(f"Saved results to {output_path}")

    # From here, you can easily convert the numerical data to a PyTorch/NumPy Tensor:
    # tensor_data = results_df[TARGET_CATEGORIES].to_numpy()

Loading model: yolo26m.pt...

PERFORMANCE STATS:
Total Images Processed: 917
Average Inference Time: 14.41 ms per image


--- Pipeline Execution Complete ---
                   filename  person  bicycle  motorcycle  car  traffic light  \
0  DJI_0408_frame_00000.jpg       1        0           0    0              0   
1  DJI_0408_frame_00001.jpg       3        0           0    0              0   
2  DJI_0408_frame_00002.jpg       0        0           0    2              0   
3  DJI_0408_frame_00003.jpg       0        0           0    0              0   
4  DJI_0408_frame_00004.jpg       0        0           0    2              0   

             timestamp  
0  2026-07-17 14:13:58  
1  2026-07-17 14:13:58  
2  2026-07-17 14:13:59  
3  2026-07-17 14:13:59  
4  2026-07-17 14:14:01  
Saved results to /content/drive/MyDrive/datasets/results.csv
